## Accessing and analysing data from the Perovskite Tandem Database
This notebook gives a short demonstration for how to access and query the perovskite Tandem database in NOMAD.

In order to access NOMAD entries, you first need to select a URL of the relevant NOMAD installation and provide a generated API token for authorization. Below, a URL for the central example Oasis is already selected. API token can be generated in GUI by a logged-in user; in case of example OASIS got to the [APIs page](https://nomad-lab.eu/oasis/gui/analyze/apis), under `App token` select desired expiration date and copy the token via the icon next to the right.

The token should be stored in a `.env` file in the project; this file is not tracked by github. Create it or copy it from the example file `.env.example`, then replace the placeholder with the actual token (without any quotation marks or other symbols).

In [6]:
import json
import os
import time

# import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

url = "https://nomad-lab.eu/oasis/backend/api/v1"
API_TOKEN = os.environ["API_TOKEN"]

### Read in a specific entry from the perovskite tandem database

In [7]:
# Read a single entry you have access to (public, created by you or shared with you)
# and export it to a JSON file. The file is selected by the entry ID

entry_id_to_export="--9Q9v_0LGFevnjJo8_PjTpwUWQr"

def load_entry(url: str, token: str, entry_id: str) -> list[dict]:
    query = {
        "required": {
            "data": "*",
        },
        "owner": "visible",
        "query": {
            "entry_id": entry_id,
            "entry_type": "PerovskiteTandemSolarCell",
        },
        "pagination": {"page_size": 50},
    }

    linked_data = []

    while True:
        response = requests.post(
            f"{url}/entries/archive/query",
            headers={"Authorization": f"Bearer {token}"},
            json=query,
        )
        response.raise_for_status()
        response_json = response.json()
        linked_data.extend(response_json["data"])

        next_value = response_json["pagination"].get("next_page_after_value")
        if not next_value:
            break
        query["pagination"]["page_after_value"] = next_value

    return linked_data

full_data = load_entry(url, API_TOKEN, entry_id_to_export)

data = full_data[0]['archive']
if "m_ref_archives" in data:
    del data["m_ref_archives"]
json_data = json.dumps(data, indent=2)

with open("result.json", "w") as f:
    f.write(json_data)

print(f"### The entry with id {entry_id_to_export} has been saved into result.json ###")

### The entry with id --9Q9v_0LGFevnjJo8_PjTpwUWQr has been saved into result.json ###


### Read in all datafiles in the perovsktie tandem database

In [8]:
def load_all_entries(url: str, token: str) -> list[dict]:
    query = {
        "required": {
            "data": "*",
        },
        "owner": "visible",
        "query": {
            "entry_type": "PerovskiteTandemSolarCell",
        },
        "pagination": {"page_size": 500},
    }

    linked_data = []

    while True:
        response = requests.post(
            f"{url}/entries/archive/query",
            headers={"Authorization": f"Bearer {token}"},
            json=query,
        )
        response.raise_for_status()
        response_json = response.json()
        linked_data.extend(response_json["data"])

        next_value = response_json["pagination"].get("next_page_after_value")
        if not next_value:
            break
        query["pagination"]["page_after_value"] = next_value
        time.sleep(3)  # Add a delay due to server rate limiting

    return linked_data

time.sleep(5)  # Add a delay due to server rate limiting
full_data = load_all_entries(url, API_TOKEN)

print(f"### Found {len(full_data)} tandem solar cell entries ###")

### Found 638 tandem solar cell entries ###


#### Extract selected parameters and convert them into a pandas dataframe

In [9]:
def get_nested(dictionary: dict, path: str, default=pd.NA):
    """Retrieve a nested value using dot-separated keys."""
    value = dictionary

    for key in path.split("."):
        if not isinstance(value, dict) or key not in value:
            return default
        value = value[key]

    return value


# Select the fields to extract into pandas table. The actual information is under archive.data
selected_fields = {
    "Entry ID": "entry_id",
    "Upload ID": "upload_id",
    "Publication": "archive.data.reference.publication_title",
    "Architecture": "archive.data.general.architecture",
    "Efficiency": "archive.data.key_performance_metrics.power_conversion_efficiency",
    "Open Circuit Voltage": "archive.data.key_performance_metrics.open_circuit_voltage",
    "Short Circuit Current Density": "archive.data.key_performance_metrics.short_circuit_current_density",
    "Fill Factor": "archive.data.key_performance_metrics.fill_factor",
}

rows = [
    {
        column_name: get_nested(entry, field_path)
        for column_name, field_path in selected_fields.items()
    }
    for entry in full_data
]

df = pd.DataFrame(rows)

print(df.shape)
display(df.head(n = 20))    # show part of the table


(638, 8)


,Entry ID,Upload ID,Publication,Architecture,Efficiency,Open Circuit Voltage,Short Circuit Current Density,Fill Factor
0,--9Q9v_0LGFevnjJo8_PjTpwUWQr,eXtCG4zSRWa3pvqJCZdbQQ,Large-Area 23%-Efficient Monolithic Perovskite...,Monolithic,20.1,1.728,14.1,0.82
1,-2u1-VnrSoWjRKHuljXg6Dhnf-Xd,eXtCG4zSRWa3pvqJCZdbQQ,CH<sub>3</sub>NH<sub>3</sub>PbBr<sub>3</sub>–C...,Laminated,10.8,1.95,8.4,0.66
2,-IhT-iHq5gSjGnh3X8sJny0CyeVR,eXtCG4zSRWa3pvqJCZdbQQ,Inverted pyramidally-textured PDMS antireflect...,Monolithic,21.93,1.75,16.89,0.742
3,-Qv-zu_a_ARgPtFZjYkptQ8s3691,eXtCG4zSRWa3pvqJCZdbQQ,A Three-Terminal Monolithic Perovskite/Si Tand...,Monolithic,22.91,1.63,17.8,0.79
4,-R9sEsVn3UikxK1W3oNksYm-FVGO,eXtCG4zSRWa3pvqJCZdbQQ,Blade-Coated Perovskites on Textured Silicon f...,Monolithic,26.2,1.82,19.2,0.75
5,-hk__5zaop_7lApesDJ4tqTo4zlo,eXtCG4zSRWa3pvqJCZdbQQ,2-Terminal CIGS-perovskite tandem cells: A lay...,Monolithic,1.5,0.7,6.5,0.35
6,-tbVgPQSc_Uozm6Oi1wEzZafQzDq,eXtCG4zSRWa3pvqJCZdbQQ,2-Terminal CIGS-perovskite tandem cells: A lay...,Monolithic,0.59,1.415,1.6,0.261
7,06YI8rhlnt4ISmn0RJR654Gvye4I,eXtCG4zSRWa3pvqJCZdbQQ,Low-Temperature Screen-Printed Metallization f...,Monolithic,22.0,1.693,17.28,0.713
8,0JIhDAE9uRL5LEAQBnBUmt3YI6iu,eXtCG4zSRWa3pvqJCZdbQQ,Nanostructured front electrodes for perovskite...,Stacked,23.9,<NA>,<NA>,<NA>
9,0Jr0IBPtJDBYU8TDQsAt1szuyF3H,eXtCG4zSRWa3pvqJCZdbQQ,All-perovskite tandem solar cells with 24.2% c...,Monolithic,24.0,1.925,15.5,0.807


### Query for all 2-terminal perovskite-perovskite devices in the database

In [10]:
# query cannot directly filter for the information under archive.data, so we load all that have 'Perovskite' at all and filter afterwards

def load_all_entries_extra_condition(url: str, token: str) -> list[dict]:
    query = {
        "required": {
            "data": "*",
        },
        "owner": "visible",
        "query": {
            "entry_type": "PerovskiteTandemSolarCell",
            "results.properties.optoelectronic.solar_cell.absorber": 'Perovskite',
        },
        "pagination": {"page_size": 500},
    }

    linked_data = []

    while True:
        response = requests.post(
            f"{url}/entries/archive/query",
            headers={"Authorization": f"Bearer {token}"},
            json=query,
        )
        response.raise_for_status()
        response_json = response.json()
        linked_data.extend(response_json["data"])

        next_value = response_json["pagination"].get("next_page_after_value")
        if not next_value:
            break
        query["pagination"]["page_after_value"] = next_value
        time.sleep(3)  # Add a delay due to server rate limiting

    return linked_data

time.sleep(5)  # Add a delay due to server rate limiting
full_data = load_all_entries_extra_condition(url, API_TOKEN)

print(f"### Found {len(full_data)} tandem solar cell Perovskite entries ###")

selected_data = []

for entry in full_data:
    if entry['archive']['data']['general']['photoabsorbers'] == ['Perovskite', 'Perovskite'] and entry['archive']['data']['general']['number_of_terminals'] == 2:
        selected_data.append(entry)

print(f"### Of them there are {len(selected_data)} Perovskite/Perovskite 2-terminal entries ###")


### Found 634 tandem solar cell Perovskite entries ###
### Of them there are 56 Perovskite/Perovskite 2-terminal entries ###
